# Armored Core 6 — End-to-End Pipeline from Raw TXT (Local)

This notebook runs the **whole detection pipeline locally** on a raw `.txt` file and,
optionally, validates candidates with the **trained relation verifier**. It mirrors the
professor's `logistics_end_to_end_from_txt_local.ipynb`:

1. detect ontology **mentions** (entity dictionary + rank-context rule)
2. infer **candidate relations** (trigger + domain/range)
3. **validate** them (trained verifier if available, otherwise domain/range rule)
4. **ground** entities to NamedIndividuals and build **triples**
5. draw the resulting **knowledge graph** and **query the verifier** on a manual claim.

Run the cells top to bottom. No GPU needed for rule mode.

In [ ]:
from pathlib import Path
import sys, json

# locate the project folders relative to this notebook (notebook/ -> nlp-detection/)
NLP = Path.cwd().resolve()
NLP = NLP if (NLP / "script").exists() else NLP.parent
SCRIPT = NLP / "script"
GEN = NLP / "generated"
sys.path.insert(0, str(SCRIPT))
print("project:", NLP)
print("has generated data:", (GEN / "ontology_interface.json").exists())

In [ ]:
# import the SAME logic used by the command-line scripts (single source of truth)
import importlib.util

def load(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m

import ac6_lib as L
e2e = load("e2e", SCRIPT / "05_run_end_to_end.py")   # reuse candidate + grounding logic
onto = e2e.onto
print("ontology individuals:", len(onto["individuals"]))

In [ ]:
# choose the input text and (optionally) a trained verifier
INPUT = NLP / "data-input" / "ac6_corpus.txt"
VERIFIER_DIR = NLP / "models" / "ac6_verifier" / "best_model"   # set None to force rule mode

verifier = None
if VERIFIER_DIR.exists():
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    tok = AutoTokenizer.from_pretrained(str(VERIFIER_DIR))
    mdl = AutoModelForSequenceClassification.from_pretrained(str(VERIFIER_DIR)); mdl.eval()
    def verifier(text):
        with torch.no_grad():
            logits = mdl(**tok(text, return_tensors="pt", truncation=True, max_length=192)).logits
        return mdl.config.id2label[int(logits.argmax(-1))] == "VALID"
    print("using TRAINED verifier")
else:
    print("verifier model not found -> rule mode (domain/range). Train it with 03_train_verifier.py")

In [ ]:
# run detection -> candidates -> validation -> grounding -> triples
triples = []; seen = set()
for si, sent in enumerate(L.sentence_split(INPUT.read_text(encoding="utf-8")), 1):
    for subj, pred, obj in e2e.candidates_for_sentence(sent):
        gs, go = e2e.ground(subj), e2e.ground(obj)
        if not gs or not go:
            continue
        if verifier is not None:
            sc = onto["primary_nature"](gs["short"]); oc = onto["primary_nature"](go["short"])
            if not verifier(e2e.verifier_text(pred, sc, oc)):
                continue
        key = (gs["short"], pred, go["short"])
        if key in seen:
            continue
        seen.add(key)
        triples.append((gs["short"], pred, go["short"]))
print("extracted triples:", len(triples))
for t in triples[:15]:
    print("  ", t)

## Knowledge graph of the extracted triples

In [ ]:
import matplotlib.pyplot as plt
import networkx as nx   # pip install networkx

def draw_kg(triples, limit=40):
    G = nx.DiGraph()
    for s, p, o in triples[:limit]:
        G.add_edge(s, o, label=p)
    plt.figure(figsize=(12, 9))
    pos = nx.spring_layout(G, k=0.9, seed=7)
    nx.draw(G, pos, with_labels=True, node_color="#cfe3ff", edge_color="#94a3b8",
            node_size=1300, font_size=8)
    nx.draw_networkx_edge_labels(G, pos, edge_labels=nx.get_edge_attributes(G, "label"), font_size=7)
    plt.title("Armored Core 6 — extracted knowledge graph (subset)"); plt.axis("off"); plt.show()

draw_kg(triples)

## Query the trained verifier on a manual claim

Ask the model whether a *(subject-class, relation, object-class)* statement is **VALID**.
This works only after you have trained the verifier (`03_train_verifier.py`).

In [ ]:
def ask_verifier(rel, subject_class, object_class):
    if verifier is None:
        return "rule mode: would be accepted iff domain/range match"
    text = e2e.verifier_text(rel, subject_class, object_class)
    return "VALID" if verifier(text) else "INVALID"

print("Character pilots ArmoredCore ->", ask_verifier("pilots", "Character", "ArmoredCore"))
print("Place      pilots ArmoredCore ->", ask_verifier("pilots", "Place", "ArmoredCore"))
print("Character  kills  Character   ->", ask_verifier("kills",  "Character", "Character"))